In [ ]:
# %% Libraries 
import os
import sys
import pandas as pd
import plotnine as p9
from pathlib import Path
import pypalettes as pp
import matplotlib.colors as mcolors
import scanpy as sc

In [ ]:
# %% Set Main Directory
MAIN_DIR_NAME = "cosmx_gray"
MAIN_DIR = next(p for p in Path.cwd().parents if (p / MAIN_DIR_NAME).exists()) / MAIN_DIR_NAME

# add it to sys.path to set it as root directory
sys.path.insert(0, str(MAIN_DIR))
os.chdir(MAIN_DIR)

# %% Results Directories
PLOTS_DIR = Path("/mnt/data/project0062/cosmx_gray/results/comb/spatial_plots/test_plotting")

## Prepare the inputs

In [ ]:
# Paths
OBJ_V = "niches"
adata_path = Path(f"data/comb/h5ad/comb-{OBJ_V}.h5ad")
polygons_path = Path("data/comb/polygons/comb-polygons-light.parquet")
id_col = "cell_id"

`create_crate`

In [ ]:
def create_crate(adata_path, polygons_path, id_col):
    
    adata = sc.read_h5ad(adata_path)
    polygons = pd.read_parquet(polygons_path)

    crate = {
        "adata": adata,
        "polygons": polygons,
        "id_col": id_col,
    }

    return crate

`subset_crate`

In [ ]:
def subset_crate(crate, filter_mask):

    id_col = crate["id_col"]
    adata = crate["adata"]
    polygons = crate["polygons"]

    cells = adata.obs.index[filter_mask]
    sub_adata = adata[cells, :].copy()
    sub_polygons = polygons.loc[polygons[id_col].isin(cells)].copy()

    sub_crate = {
        "adata": sub_adata,
        "polygons": sub_polygons,
        "id_col": id_col
    }
    
    return sub_crate

**Subsetting the crate**


Because cell names / barcodes are supposed to be the index of `comb.obs`, we will just get the index names of the subset.

Example code:

```{python}
filter = obj.obs["sample_name"] == "CTRL01"
sample1_cells = obj.obs.index[filter]
```

`plot_polygons`

In [ ]:
# function
def plot_polygons(
    crate, 
    ann_var, 
    ann_type=["meta"], # implement funcitonality for gene later
    fig_size:tuple=(20,20),
    ):

    meta_df = crate["adata"].obs[ann_var].copy()
    poly_df = crate["polygons"]
    id_col = crate["id_col"]


    #### map ann_var to polygons ####
    ann_poly = pd.merge(poly_df, meta_df, on=id_col, how='left')

    #### Palette ####
    # Load the colormap
    cmap = pp.load_cmap("alphabet")
    # Get N discrete colors as hex
    # use leiden_scVI because it more clusters, keeps colouring consistent
    n = len(ann_poly[ann_var].unique()) 
    palette = [cmap(i / (n-1)) for i in range(n)]
    # Convert RGBA -> hex
    palette = [mcolors.to_hex(c) for c in palette]
    
    #### plot ####
    p = (
        p9.ggplot(
            ann_poly, 
            p9.aes(
                x="x", 
                y="y", 
                group=id_col, 
                fill=ann_var
            )
        )
        + p9.geom_polygon(color="white", size=0.1)
        + p9.coord_fixed(1)  # keep aspect ratio
        + p9.scale_fill_manual(values=palette)
        + p9.theme(
            axis_line=p9.element_blank(),
            axis_text=p9.element_blank(),
            axis_ticks=p9.element_blank(),
            axis_title=p9.element_blank(),
            panel_background=p9.element_rect(fill="black"),
            panel_grid_major=p9.element_blank(),
            panel_grid_minor=p9.element_blank(), 
            figure_size=fig_size,
        )
    )

    return p
    

Example of use:

```
p = plot_polygons(
    crate = CTRL01_crate,
    ann_var = "novae_domains_8",
    fig_size = (20,20)
)

for ext in ['png', 'svg']:
p.save(
        filename= PLOTS_DIR / f'test.{ext}',
        dpi=900,
        units='in'
    )
```

## Copy this in spatial plots scripts

In [ ]:
# %% Libraries 
import os
import sys
import pandas as pd
import plotnine as p9
from pathlib import Path
import pypalettes as pp
import matplotlib.colors as mcolors
import scanpy as sc

# %% create crate
def create_crate(adata_path, polygons_path, id_col):
    
    adata = sc.read_h5ad(adata_path)
    polygons = pd.read_parquet(polygons_path)

    crate = {
        "adata": adata,
        "polygons": polygons,
        "id_col": id_col,
    }

    return crate


# %% subset crate
def subset_crate(crate, filter_mask):

    id_col = crate["id_col"]
    adata = crate["adata"]
    polygons = crate["polygons"]

    cells = adata.obs.index[filter_mask]
    sub_adata = adata[cells, :].copy()
    sub_polygons = polygons.loc[polygons[id_col].isin(cells)].copy()

    sub_crate = {
        "adata": sub_adata,
        "polygons": sub_polygons,
        "id_col": id_col
    }
    
    return sub_crate

# %% plot_polygons
def plot_polygons(
    crate, 
    ann_var, 
    ann_type=["meta"], # implement funcitonality for gene later
    fig_size:tuple=(20,20),
    ):

    meta_df = crate["adata"].obs[ann_var].copy()
    poly_df = crate["polygons"]
    id_col = crate["id_col"]


    #### map ann_var to polygons ####
    ann_poly = pd.merge(poly_df, meta_df, on=id_col, how='left')

    #### Palette ####
    # Load the colormap
    cmap = pp.load_cmap("alphabet")
    # Get N discrete colors as hex
    # use leiden_scVI because it more clusters, keeps colouring consistent
    n = len(ann_poly[ann_var].unique()) 
    palette = [cmap(i / (n-1)) for i in range(n)]
    # Convert RGBA -> hex
    palette = [mcolors.to_hex(c) for c in palette]
    
    #### plot ####
    p = (
        p9.ggplot(
            ann_poly, 
            p9.aes(
                x="x", 
                y="y", 
                group=id_col, 
                fill=ann_var
            )
        )
        + p9.geom_polygon(color="white", size=0.1)
        + p9.coord_fixed(1)  # keep aspect ratio
        + p9.scale_fill_manual(values=palette)
        + p9.theme(
            axis_line=p9.element_blank(),
            axis_text=p9.element_blank(),
            axis_ticks=p9.element_blank(),
            axis_title=p9.element_blank(),
            panel_background=p9.element_rect(fill="black"),
            panel_grid_major=p9.element_blank(),
            panel_grid_minor=p9.element_blank(), 
            figure_size=fig_size,
        )
    )

    return p
    